In [7]:
CREATE_CSV_FILES = True

TRAIN_CSV_FILE = "train.csv"
TEST_CSV_FILE = "test.csv"
#create csv file

In [8]:
import matplotlib.pyplot as plt
import numpy as np
import librosa
import csv
import os

def extractFeatures(soundFilesFolder, csvFileName): #extract the necessary features and save it in the csv file
    header ='filename rmse spectral_centroid spectral_bandwith rolloff zero_crossing_rate'
    for i in range(1, 21):
        header += f' mfcc{i}'
    header += ' label'
    header = header.split()
    file = open(csvFileName, 'w', newline='')
    writer = csv.writer(file)
    writer.writerow(header)

    for filename in os.listdir(soundFilesFolder):
        number = f'{soundFilesFolder}/{filename}'
        rmse = librosa.feature.rms(y=y)
        spec_cent = librosa.feature.spectral.spectral_centroid(y=y, sr=sr)
        spec_bw = librosa.feature.spectral.spectral_bandwidth(y=y, sr=sr)
        rolloff = librosa.feature.spectral.spectral_rolloff(y=y, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(y)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        to_append = f'{filename} {np.mean(rmse)} {np.mean(spec_cent)} {np.mean(spec_bw)} {np.mean(rolloff)} {np.mean(zcr)}'
        for e in mfcc:
            to_append += f' {np.mean(e)}'
        writer.writerow(to_append.split())
    file.close()

if (CREATE_CSV_FILES == True):
    extractFeatures("data/train", TRAIN_CSV_FILE)
    extractFeatures("data/test", TEST_CSV_FILE)
else:
    print("CSV files creation is skipped")

data/train/20240409_204306.mp3
data/train/20240409_204310.mp3
data/train/20240409_204315.mp3
data/train/20240409_204319.mp3
data/train/20240409_204322.mp3
data/train/20240409_204410.mp3
data/train/20240409_204414.mp3
data/train/20240409_204417.mp3
data/train/20240409_204420.mp3
data/train/20240409_204423.mp3
data/test/20240409_204426.mp3
data/test/20240409_204428.mp3
data/test/20240409_204430.mp3


In [26]:
import pandas as pd
import csv
from sklearn import preprocessing

def preProcessData(csvFileName): #stratifying the dataset, adding "0" label if they're the speaker, removing uncessary rows for processing such as filename
    data = pd.read_csv(csvFileName)
    filenameArray = data['filename']
    speakerArray = []
    for i in range(len(filenameArray)):
        speaker = "0"
        speakerArray.append(speaker)
    data['number'] = speakerArray

    data = data.drop(['filename'], axis=1)
    data = data.drop(['label'], axis=1)
    data.shape
    return data

trainData = preProcessData(TRAIN_CSV_FILE)
testData = preProcessData(TEST_CSV_FILE)

print(trainData)

       rmse  spectral_centroid  spectral_bandwith      rolloff  \
0  0.084662        1618.111898        2064.846379  3214.780561   
1  0.126201        1352.913842        1951.914482  2792.138672   
2  0.121898        1675.670111        1932.109531  3204.279549   
3  0.072539        2365.903388        2238.296052  4216.133881   
4  0.118728        1541.656324        1961.821521  2816.184082   
5  0.121288        1182.990433        1877.968249  2269.360352   
6  0.126577        1212.409459        1768.293967  2272.803330   
7  0.101003        1632.915479        1984.859526  3020.256042   
8  0.140247        1912.230941        1942.019296  3297.584711   
9  0.160155        1103.405178        1774.817386  2154.858398   

   zero_crossing_rate       mfcc1       mfcc2      mfcc3      mfcc4  \
0            0.059858 -217.201309  126.457588  12.106718  17.929337   
1            0.049931 -201.773407  134.614304  22.523781  31.559008   
2            0.088977 -210.061813  108.051125  24.383257  53

In [27]:
from sklearn.model_selection import train_test_split
x = np.array(trainData.iloc[:, :-1], dtype=float)
y = trainData.iloc[:, -1]
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42)

x_test = np.array(testData.iloc[:, :-1], dtype=float)
y_test = testData.iloc[:, -1]

#print(testData)
print(y_train)
print(x_train.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

0    0
7    0
2    0
9    0
4    0
3    0
6    0
Name: number, dtype: object
(7, 25)
(7,)
(3,)
(3,)


In [28]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
scaler = StandardScaler()
x_train = scaler.fit_transform( x_train )
x_val = scaler.transform( x_val )
x_test = scaler.transform( x_test )

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

(7, 25)
(3, 25)
(3, 25)


In [29]:
from keras import models
from keras import layers
import keras

model = models.Sequential()
model.add(layers.Dense(256, activation='relu', input_shape=(x_train.shape[1],)))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(10, activation='sigmoid'))

model.compile(optimizer = 'adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

'''model.add(layers.Convolution2D(7, (3,3), activation='relu', input_shape=(7,25,1)))
model.add(layers.Convolution2D(7, (3,3), activation='relu'))
model.compile(optimizer='adam', loss='BinaryCrossentropy', metrics=['accuracy'])
model.summary() history = model.fit(x_train, validation_data=(x_val, y_val), epochs=4)'''

from keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1)
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=4, batch_size=128, callbacks=[es])

from matplotlib import pyplot
pyplot.plot(history.history['loss'], label='train')
pyplot.plot(history.history['val_loss'], label='test')
pyplot.legend()
pyplot.show()

AttributeError: module 'keras.src.backend' has no attribute 'floatx'

In [15]:
from matplotlib import pyplot
pyplot.plot(history.history['loss'], label='train')
pyplot.plot(history.history['val_loss'], label='test')
pyplot.legend()
pyplot.show()

NameError: name 'history' is not defined

In [17]:
def getSpeaker(speaker):
    speaker = str(speaker)
    if speaker == "0":
        return "Nice"
    else:
        return "Unknown"

def printPrediction(x_data, y_data, printDigit):
    for i in range(len(y_data)):
        predictions = getSpeaker(model.predict(x_data[i:i+1]))
        prediction = getSpeaker(model.predict(x_data[i:i+1])[0])
        speaker = getSpeaker(y_data[i])
        if printDigit == True:
            print("Number ={0:d}, y={1:10s}- prediction={2:10s}- match={3}".format(i, speaker, prediction, speaker==prediction))
        else:
            print("y={0:10s}- prediction ={1:10s}- match={2}".format(speaker, prediction, speaker==prediction))

In [18]:
import numpy as np
from keras import backend as K
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Convolution2D, MaxPooling2D
from keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

def report(x_data, y_data):
    y_pred = model.predict_classes(x_data)
    y_test_num = y_data.astype(np.int64)
    conf_mt = confusion_matrix(y_test_num, y_pred)
    print(conf_mt)
    plt.matshow(conf_mt)
    print('\nClassification Report')
    target_names = ["Nice", 'Unknown']
    print(classification_report(y_test_num, y_pred))


In [19]:
print('\n# TEST DATA #\n')
score = model.evaluate(x_test, y_test)
print("%s: %.2f%%" % (model.metrics_names[1], score[1]*100))

printPrediction(x_test[0:10], y_test[0:3], True)


# TEST DATA #



ValueError: in user code:

    File "c:\Users\daren\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py", line 2066, in test_function  *
        return step_function(self, iterator)
    File "c:\Users\daren\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py", line 2049, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\daren\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py", line 2037, in run_step  **
        outputs = model.test_step(data)
    File "c:\Users\daren\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py", line 1917, in test_step
        y_pred = self(x, training=False)
    File "c:\Users\daren\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    F